# IND-3 : Défauts synthétiques par diffusion pour l'inspection visuelle

Question de recherche : l'ajout de défauts générés par diffusion améliore-t-il la détection de défauts rares, ou ne fait-il que dupliquer les cas déjà présents ?

## Cellule 1 — Imports + constantes

In [11]:
import os
import random
import shutil
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from torchvision.transforms import v2 as T
from tqdm import tqdm
from lightning.pytorch.callbacks import TQDMProgressBar

from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.metrics import AUPRO, AUROC, F1Score
from anomalib.metrics.evaluator import Evaluator
from anomalib.models import Patchcore

# Constantes
SEEDS = [42, 43, 44]
BATCH_SIZE = 32
IMAGE_SIZE = 256
CATEGORIES = ["bottle", "cable", "transistor"]
REGIMES = ["R1_reel_seul", "R2_aug_classique", "R3_synthetique"]
DATA_DIR = "./data"
RESULTS_DIR = "./results"
FIGURES_DIR = "./figures"
SYNTH_DIR = "./synthetiques"

COLONNES_RESULTS = [
    "timestamp",
    "categorie",
    "regime",
    "seed",
    "auroc_image",
    "auroc_pixel",
    "aupro"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print("Device :", DEVICE)
print("Catégories :", CATEGORIES)
print("Régimes :", REGIMES)


Device : cuda
Catégories : ['bottle', 'cable', 'transistor']
Régimes : ['R1_reel_seul', 'R2_aug_classique', 'R3_synthetique']


## Cellule 2 — Téléchargement et préparation des données

In [12]:
def creer_datamodule(categorie, batch_size, image_size, data_dir, regime="R1_reel_seul", synth_dir=None):
    """Crée le datamodule MVTec AD pour une catégorie et un régime donnés."""
    train_augmentations = None
    if regime == "R2_aug_classique":
        train_augmentations = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.GaussianNoise(mean=0.0, sigma=0.1),
        ])

    datamodule = MVTecAD(
        root=data_dir,
        category=categorie,
        train_batch_size=batch_size,
        eval_batch_size=batch_size,
        num_workers=0,
        train_augmentations=train_augmentations,
    )
    datamodule.prepare_data()
    datamodule.setup()

    if regime == "R3_synthetique":
        if synth_dir is None:
            raise ValueError("synth_dir est requis pour le régime R3_synthetique")
        injecter_synthetiques_dans_train(datamodule, categorie, synth_dir)

    return datamodule


def afficher_counts_datamodule(datamodule, categorie):
    """Affiche les nombres d'images par split pour une catégorie."""
    n_train = len(datamodule.train_data) if hasattr(datamodule, "train_data") else 0
    n_test = len(datamodule.test_data) if hasattr(datamodule, "test_data") else 0
    n_val = len(datamodule.val_data) if hasattr(datamodule, "val_data") else 0
    print(f"--- Counts pour {categorie} ---")
    print(f"  Train : {n_train}")
    print(f"  Test  : {n_test}")
    print(f"  Val   : {n_val}")


for cat in CATEGORIES:
    dm = creer_datamodule(cat, BATCH_SIZE, IMAGE_SIZE, DATA_DIR)
    afficher_counts_datamodule(dm, cat)


--- Counts pour bottle ---
  Train : 209
  Test  : 83
  Val   : 83
--- Counts pour cable ---
  Train : 224
  Test  : 150
  Val   : 150
--- Counts pour transistor ---
  Train : 213
  Test  : 100
  Val   : 100


## Cellule 3 — Fonctions utilitaires

In [13]:
def set_seed(seed):
    """Fixe les graines pour la reproductibilité."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sauvegarder_resultat(ligne_dict, chemin_csv):
    """Ajoute une ligne de résultats dans le CSV selon le schéma fixé."""
    ligne_dict["timestamp"] = datetime.now().isoformat()
    df = pd.DataFrame([ligne_dict])
    if os.path.exists(chemin_csv):
        df.to_csv(chemin_csv, mode="a", header=False, index=False)
    else:
        df.to_csv(chemin_csv, mode="w", header=True, index=False)
    print(f"Résultat sauvegardé : {ligne_dict}")


def sauvegarder_figure(fig, nom):
    """Sauvegarde une figure matplotlib en PNG 150 dpi."""
    chemin = os.path.join(FIGURES_DIR, nom)
    fig.savefig(chemin, dpi=150, bbox_inches="tight")
    print(f"Figure sauvegardée : {chemin}")
    plt.close(fig)

## Cellule 4 — Entraînement et évaluation

In [14]:
def compter_images_synthetiques(synth_dir, categorie):
    """Compte et affiche le nombre d'images synthétiques par catégorie."""
    chemin = os.path.join(synth_dir, categorie)
    if not os.path.exists(chemin):
        print(f"[!] Pas de dossier synthétique pour {categorie} : {chemin}")
        return 0
    images = [f for f in os.listdir(chemin) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    print(f"Synthétiques {categorie} : {len(images)} images")
    return len(images)


def ajouter_synthetiques_au_train(datamodule, chemins):
    """Ajoute des images synthétiques comme échantillons normaux du train."""
    chemins = [Path(p) for p in chemins]
    if not chemins:
        return 0
    train_data = datamodule.train_data
    samples = train_data.samples.copy()
    extra = pd.DataFrame({
        "path": [str(p.parent) for p in chemins],
        "split": ["train"] * len(chemins),
        "label": [p.parent.name for p in chemins],
        "image_path": [str(p) for p in chemins],
        "mask_path": [None] * len(chemins),
        "label_index": [0] * len(chemins),
    })
    nouveaux = pd.concat([samples, extra], ignore_index=True)
    nouveaux.attrs = dict(samples.attrs)
    train_data.samples = nouveaux
    train_data.samples.attrs = dict(samples.attrs)
    print(f"Ajouté {len(chemins)} images synthétiques au train de {datamodule.category}")
    return len(chemins)


def injecter_synthetiques_dans_train(datamodule, categorie, synth_dir):
    """Ajoute les images synthétiques de synth_dir/<catégorie> au train du datamodule."""
    src = Path(synth_dir, categorie)
    if not src.exists():
        raise FileNotFoundError(f"Dossier synthétique manquant : {src}")
    chemins = sorted(
        p for p in src.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )
    return ajouter_synthetiques_au_train(datamodule, chemins)


def sauvegarder_predictions(predictions, chemin_csv):
    """Sauvegarde les scores et labels du test dans un CSV."""
    lignes = []
    for batch in predictions:
        for item in batch:
            if item.image_path is None:
                continue
            label = item.gt_label
            score = item.pred_score
            if isinstance(label, torch.Tensor):
                label = label.item()
            if isinstance(score, torch.Tensor):
                score = score.item()
            lignes.append({"image_path": str(item.image_path), "label": label, "score": score})
    pd.DataFrame(lignes).to_csv(chemin_csv, index=False)
    print(f"Prédictions sauvegardées : {chemin_csv}")


def creer_evaluateur():
    """Évaluateur avec AUROC/F1 image et pixel, plus AUPRO pixel."""
    return Evaluator(
        test_metrics=[
            AUROC(fields=["pred_score", "gt_label"], prefix="image_"),
            F1Score(fields=["pred_label", "gt_label"], prefix="image_"),
            AUROC(fields=["anomaly_map", "gt_mask"], prefix="pixel_", strict=False),
            F1Score(fields=["pred_mask", "gt_mask"], prefix="pixel_", strict=False),
            AUPRO(fields=["anomaly_map", "gt_mask"], prefix="pixel_", strict=False),
        ],
    )


def entrainer_et_evaluer(regime, seed, categorie):
    """Entraîne PatchCore, évalue et sauvegarde les métriques et prédictions."""
    set_seed(seed)
    print(f"\n=== {regime} | {categorie} | seed={seed} ===")

    datamodule = creer_datamodule(
        categorie, BATCH_SIZE, IMAGE_SIZE, DATA_DIR, regime=regime, synth_dir=SYNTH_DIR
    )

    model = Patchcore(evaluator=creer_evaluateur())
    engine = Engine(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        max_epochs=1,
        enable_progress_bar=True,
        callbacks=[TQDMProgressBar()],
    )

    engine.fit(model=model, datamodule=datamodule)
    results = engine.test(model=model, datamodule=datamodule)

    auroc_image = float(results[0].get("image_AUROC", float("nan")))
    auroc_pixel = float(results[0].get("pixel_AUROC", float("nan")))
    aupro = float(results[0].get("pixel_AUPRO", float("nan")))

    ligne = {
        "categorie": categorie,
        "regime": regime,
        "seed": seed,
        "auroc_image": auroc_image,
        "auroc_pixel": auroc_pixel,
        "aupro": aupro
    }
    sauvegarder_resultat(ligne, os.path.join(RESULTS_DIR, "results.csv"))

    predictions = engine.predict(model=model, datamodule=datamodule)
    if predictions:
        pred_path = os.path.join(RESULTS_DIR, f"predictions_{regime}_{categorie}_{seed}.csv")
        sauvegarder_predictions(predictions, pred_path)

    return ligne


## Cellule 5 — Entraînement principal (cellule coûteuse, exécutable seule)

In [15]:
def lire_combinaisons_deja_faites(chemin_csv):
    """Lit les combinaisons (catégorie, régime, graine) déjà effectuées."""
    if not os.path.exists(chemin_csv):
        return set()
    df = pd.read_csv(chemin_csv)
    if df.empty:
        return set()
    return set(zip(df["categorie"], df["regime"], df["seed"].astype(int)))


def executer_experiences():
    """Boucle principale idempotente : seeds × catégories × régimes."""
    chemin_csv = os.path.join(RESULTS_DIR, "results.csv")
    deja_faites = lire_combinaisons_deja_faites(chemin_csv)
    total = len(SEEDS) * len(CATEGORIES) * len(REGIMES)
    print(f"Combinations déjà faites : {len(deja_faites)} / {total}")

    for seed in SEEDS:
        for categorie in CATEGORIES:
            for regime in REGIMES:
                if (categorie, regime, seed) in deja_faites:
                    print(f"Skip {regime} | {categorie} | seed={seed}")
                    continue
                if regime == "R3_synthetique":
                    dossier = Path(SYNTH_DIR, categorie)
                    images = [q for q in dossier.glob("*") if q.suffix.lower() in {".png", ".jpg", ".jpeg"}] if dossier.exists() else []
                    if not images:
                        print(f"[!] R3 ignoré | {categorie} | seed={seed} : aucun synthétique dans {dossier}")
                        continue

                try:
                    entrainer_et_evaluer(regime, seed, categorie)
                except Exception as e:
                    print(f"ERREUR {regime} | {categorie} | seed={seed} : {e}")

executer_experiences()

Combinations déjà faites : 16 / 27
Skip R1_reel_seul | bottle | seed=42
Skip R2_aug_classique | bottle | seed=42
Skip R3_synthetique | bottle | seed=42
Skip R1_reel_seul | cable | seed=42
Skip R2_aug_classique | cable | seed=42
Skip R3_synthetique | cable | seed=42
Skip R1_reel_seul | transistor | seed=42
Skip R2_aug_classique | transistor | seed=42
Skip R3_synthetique | transistor | seed=42
Skip R1_reel_seul | bottle | seed=43
Skip R2_aug_classique | bottle | seed=43
Skip R3_synthetique | bottle | seed=43
Skip R1_reel_seul | cable | seed=43
Skip R2_aug_classique | cable | seed=43
Skip R3_synthetique | cable | seed=43
Skip R1_reel_seul | transistor | seed=43

=== R2_aug_classique | transistor | seed=43 ===


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 20                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=59` in the `DataLoader` to improve performance.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=59` in the `DataLoader` to improve performance.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnost

Training: |          | 0/? [00:00<?, ?it/s]

... (145 outputs left out)

Résultat sauvegardé : {'categorie': 'transistor', 'regime': 'R3_synthetique', 'seed': 44, 'auroc_image': 0.9954165816307068, 'auroc_pixel': 0.9739673733711243, 'aupro': 0.947937548160553, 'timestamp': '2026-11-14T14:39:35.155278'}


/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=59` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Prédictions sauvegardées : ./results/predictions_R3_synthetique_transistor_44.csv


## Cellule 6 — Agrégation des résultats

In [16]:
def agreger_resultats(results_csv, summary_csv):
    """Calcule moyenne ± écart-type par régime et par catégorie/régime."""
    df = pd.read_csv(results_csv)
    grouped = df.groupby(["categorie", "regime"])[["auroc_image", "auroc_pixel", "aupro"]]
    summary = grouped.agg(["mean", "std"]).reset_index()
    summary.to_csv(summary_csv, index=False)
    print(summary)
    return summary


summary = agreger_resultats(
    os.path.join(RESULTS_DIR, "results.csv"),
    os.path.join(RESULTS_DIR, "summary.csv")
)

    categorie            regime auroc_image           auroc_pixel            \
                                       mean       std        mean       std   
0      bottle      R1_reel_seul    1.000000  0.000000    0.985600  0.000044   
1      bottle  R2_aug_classique    0.994643  0.001875    0.977286  0.001816   
2      bottle    R3_synthetique    1.000000  0.000000    0.985600  0.000044   
3       cable      R1_reel_seul    0.987772  0.003783    0.984921  0.000233   
4       cable  R2_aug_classique    0.975590  0.001031    0.972327  0.000778   
5       cable    R3_synthetique    0.987772  0.003783    0.984921  0.000233   
6  transistor      R1_reel_seul    0.994479  0.001377    0.973412  0.000647   
7  transistor  R2_aug_classique    0.941917  0.003443    0.920763  0.004356   
8  transistor    R3_synthetique    0.994500  0.001193    0.973322  0.000595   

      aupro            
       mean       std  
0  0.944900  0.000310  
1  0.909482  0.006764  
2  0.944900  0.000310  
3  0.93126

In [17]:
import os


def _executer_cellules_amont():
    """Rejoue les cellules amont nécessaires si la cellule est lancée seule."""
    import json
    from IPython import get_ipython
    ip = get_ipython()
    if ip is None:
        return
    chemin_nb = "notebook_IND3_diffusion_defauts.ipynb"
    if not os.path.exists(chemin_nb):
        return
    marqueurs = ["SEEDS =", "def creer_datamodule", "def set_seed", "def creer_evaluateur"]
    faits = set()
    with open(chemin_nb, encoding="utf-8") as f:
        cellules = json.load(f)["cells"]
    for cellule in cellules:
        if cellule["cell_type"] != "code":
            continue
        source = "".join(cellule["source"])
        if "def executer_experiences" in source or "def executer_ablations_aug" in source:
            continue
        for marqueur in marqueurs:
            if marqueur in source and marqueur not in faits:
                ip.run_cell(source, store_history=False)
                faits.add(marqueur)


if "creer_evaluateur" not in globals() or "creer_datamodule" not in globals():
    print("Bootstrap : exécution des cellules amont nécessaires...")
    _executer_cellules_amont()


def executer_ablations_aug(categorie, seed):
    """Teste R2 en retirant successivement flip, rotation, bruit."""
    set_seed(seed)
    configs = {
        "flip_seul": T.RandomHorizontalFlip(p=0.5),
        "rotation_seule": T.RandomRotation(degrees=15),
        "bruit_seul": T.GaussianNoise(mean=0.0, sigma=0.1),
    }
    lignes = []
    for nom, augmentation in configs.items():
        print(f"\n--- Ablations R2 : {nom} ---")
        datamodule = MVTecAD(
            root=DATA_DIR, category=categorie,
            train_batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE,
            num_workers=0, train_augmentations=augmentation,
        )
        datamodule.prepare_data()
        datamodule.setup()
        model = Patchcore(evaluator=creer_evaluateur())
        engine = Engine(
            accelerator="gpu" if torch.cuda.is_available() else "cpu",
            devices=1,
            max_epochs=1,
            callbacks=[TQDMProgressBar()],
        )
        engine.fit(model=model, datamodule=datamodule)
        results = engine.test(model=model, datamodule=datamodule)
        lignes.append({
            "categorie": categorie, "seed": seed, "ablation": nom,
            "auroc_image": float(results[0].get("image_AUROC", float("nan"))),
            "auroc_pixel": float(results[0].get("pixel_AUROC", float("nan"))),
            "aupro": float(results[0].get("pixel_AUPRO", float("nan")))
        })
    df = pd.DataFrame(lignes)
    df.to_csv(os.path.join(RESULTS_DIR, "ablations_aug.csv"), index=False)
    return df


def filtrer_synthetiques_quasi_dupliques(categorie, synth_dir, data_dir, results_dir):
    """Retire les synthétiques sous la médiane des distances au vrai défaut."""
    distances_csv = os.path.join(results_dir, f"distances_quasi_duplication_{categorie}.csv")
    df = pd.read_csv(distances_csv)
    seuil = df["distance"].median()
    a_garder = df[df["distance"] >= seuil]["synthetique"].tolist()
    print(f"Filtrage {categorie} : garde {len(a_garder)} / {len(df)} (seuil médiane={seuil:.4f})")
    return a_garder, seuil


def executer_ablation_synth(categorie, seed):
    """Entraîne R3 sans les synthétiques quasi-dupliqués."""
    set_seed(seed)
    print(f"\n--- Ablations R3 : filtrage quasi-dupliqués | {categorie} | seed={seed} ---")

    a_garder, seuil = filtrer_synthetiques_quasi_dupliques(categorie, SYNTH_DIR, DATA_DIR, RESULTS_DIR)
    datamodule = MVTecAD(
        root=DATA_DIR, category=categorie,
        train_batch_size=BATCH_SIZE, eval_batch_size=BATCH_SIZE,
        num_workers=0,
    )
    datamodule.prepare_data()
    datamodule.setup()
    ajouter_synthetiques_au_train(datamodule, a_garder)

    model = Patchcore(evaluator=creer_evaluateur())
    engine = Engine(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        max_epochs=1,
        callbacks=[TQDMProgressBar()],
    )
    engine.fit(model=model, datamodule=datamodule)
    results = engine.test(model=model, datamodule=datamodule)
    ligne = {
        "categorie": categorie, "seed": seed, "ablation": "synth_filtree", "seuil": seuil,
        "auroc_image": float(results[0].get("image_AUROC", float("nan"))),
        "auroc_pixel": float(results[0].get("pixel_AUROC", float("nan"))),
        "aupro": float(results[0].get("pixel_AUPRO", float("nan")))
    }
    df = pd.DataFrame([ligne])
    df.to_csv(os.path.join(RESULTS_DIR, "ablations_synth.csv"), index=False)
    return df


def fig10_barplot_ablations(df_aug, df_synth):
    """Barplot comparant les ablations data-aug et filtrage synthétique."""
    frames = []
    if df_aug is not None and not df_aug.empty:
        tmp = df_aug.copy()
        tmp["type"] = "aug"
        frames.append(tmp[["ablation", "auroc_image", "type"]])
    if df_synth is not None and not df_synth.empty:
        tmp = df_synth.copy()
        tmp["type"] = "synth"
        frames.append(tmp[["ablation", "auroc_image", "type"]])
    if not frames:
        print("[!] Aucune ablation à tracer.")
        return
    df = pd.concat(frames, ignore_index=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=df, x="ablation", y="auroc_image", hue="type", ax=ax)
    ax.set_ylim([0, 1])
    ax.set_title("Ablations : impact sur l'AUROC image")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig10_ablations.png")


cat_ref = CATEGORIES[0]
seed_ref = SEEDS[0]

df_aug = None
try:
    df_aug = executer_ablations_aug(cat_ref, seed_ref)
except Exception as e:
    print(f"[!] Ablations data-aug ignorées : {e}")

df_synth = None
distances_csv = os.path.join(RESULTS_DIR, f"distances_quasi_duplication_{cat_ref}.csv")
if Path(SYNTH_DIR, cat_ref).exists() and os.path.exists(distances_csv):
    try:
        df_synth = executer_ablation_synth(cat_ref, seed_ref)
    except Exception as e:
        print(f"[!] Ablation synthétique ignorée : {e}")
else:
    print(f"[!] Ablation synthétique ignorée : aucun synthétique/distance pour {cat_ref}")

fig10_barplot_ablations(df_aug, df_synth)



--- Ablations R2 : flip_seul ---


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 20                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=59` in the `DataLoader` to improve performance.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnostic/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=59` in the `DataLoader` to improve performance.
/home/moussnaoui/PycharmProjects/DIffusion-Diagnost

Training: |          | 0/? [00:00<?, ?it/s]

... (31 outputs left out)



Selecting Coreset Indices.:   0%|          | 0/21400 [00:00<?, ?it/s]

Selecting Coreset Indices.:   0%|          | 53/21400 [00:00<00:40, 525.33it/s]

Selecting Coreset Indices.:   1%|          | 113/21400 [00:00<00:37, 566.41it/s]

Selecting Coreset Indices.:   1%|          | 173/21400 [00:00<00:36, 578.67it/s]

Selecting Coreset Indices.:   1%|          | 233/21400 [00:00<00:36, 585.00it/s]

Selecting Coreset Indices.:   1%|▏         | 293/21400 [00:00<00:35, 588.80it/s]

Selecting Coreset Indices.:   2%|▏         | 353/21400 [00:00<00:35, 590.84it/s]

Selecting Coreset Indices.:   2%|▏         | 413/21400 [00:00<00:35, 592.05it/s]

Selecting Coreset Indices.:   2%|▏         | 473/21400 [00:00<00:44, 470.39it/s]

Selecting Coreset Indices.:   2%|▏         | 533/21400 [00:00<00:41, 503.46it/s]

Selecting Coreset Indices.:   3%|▎         | 593/21400 [00:01<00:39, 528.65it/s]

Selecting Coreset Indices.:   3%|▎         | 653/21400 [00:01<00:37, 547.43it/s]

Selecting Coreset Indices.

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            1.0            │
│       image_F1Score       │    0.9919999837875366     │
│        pixel_AUPRO        │    0.9452453851699829     │
│        pixel_AUROC        │    0.9856631755828857     │
│       pixel_F1Score       │    0.7272874116897583     │
└───────────────────────────┴───────────────────────────┘

Figure sauvegardée : ./figures/fig10_ablations.png


## Cellule 7 — Figures pour le rapport

In [18]:
def fig1_barplot_auroc_regime(df):
    """Barplot de l'AUROC image moyen par régime avec écart-type."""
    agg = df.groupby("regime")["auroc_image"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(agg["regime"], agg["mean"], yerr=agg["std"], capsize=5)
    ax.set_ylabel("AUROC image")
    ax.set_title("AUROC image moyen par régime")
    ax.set_ylim([0, 1])
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig1_auroc_par_regime.png")


def fig2_barplot_aupro_regime(df):
    """Barplot de l'AUPRO moyen par régime avec écart-type."""
    agg = df.groupby("regime")["aupro"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(agg["regime"], agg["mean"], yerr=agg["std"], capsize=5)
    ax.set_ylabel("AUPRO")
    ax.set_title("AUPRO moyen par régime")
    ax.set_ylim([0, 1])
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig2_aupro_par_regime.png")


def fig3_heatmap_auroc(df):
    """Heatmap AUROC (catégories × régimes)."""
    pivot = df.pivot_table(values="auroc_image", index="categorie", columns="regime", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title("AUROC image moyen : catégories × régimes")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig3_heatmap_auroc.png")


def fig4_barplot_auroc_par_graine(df, regime):
    """Barplot des AUROC par graine pour un régime donné."""
    sub = df[df["regime"] == regime]
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=sub, x="categorie", y="auroc_image", hue="seed", ax=ax)
    ax.set_title(f"AUROC image par graine — {regime}")
    ax.set_ylim([0, 1])
    plt.legend(title="Seed")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig4_auroc_par_graine_{regime}.png")


def fig5_grille_exemples(categorie, synth_dir, data_dir, n_col=5):
    """Grille 3 lignes : normales, défauts réels, défauts synthétiques."""
    chemins_normaux = sorted(Path(data_dir, categorie, "train", "good").glob("*"))[:n_col]
    chemins_defauts = []
    test_dir = Path(data_dir, categorie, "test")
    for d in test_dir.iterdir():
        if d.is_dir() and d.name != "good":
            chemins_defauts.extend(sorted(d.glob("*")))
            if len(chemins_defauts) >= n_col:
                break
    chemins_defauts = chemins_defauts[:n_col]
    chemins_synth = sorted(Path(synth_dir, categorie).glob("*"))[:n_col]

    fig, axes = plt.subplots(3, n_col, figsize=(n_col * 2, 6))
    titres = ["Normaux", "Défauts réels", "Défauts synthétiques"]
    listes = [chemins_normaux, chemins_defauts, chemins_synth]
    for i, (titre, liste) in enumerate(zip(titres, listes)):
        axes[i, 0].set_ylabel(titre)
        for j in range(n_col):
            ax = axes[i, j]
            ax.axis("off")
            if j < len(liste):
                img = Image.open(liste[j]).convert("RGB")
                ax.imshow(img)
    plt.suptitle(f"Exemples — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig5_exemples_{categorie}.png")


def fig6_analyse_erreurs(predictions_csv, n_exemples=10):
    """Affiche les exemples de test les plus mal classés."""
    df = pd.read_csv(predictions_csv)
    df["erreur"] = np.abs(df["label"] - df["score"])
    pires = df.sort_values("erreur", ascending=False).head(n_exemples)
    n_col = min(n_exemples, 5)
    n_row = int(np.ceil(n_exemples / n_col))
    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 2, n_row * 2))
    axes = np.array(axes).flatten()
    for idx, (_, row) in enumerate(pires.iterrows()):
        if idx >= len(axes):
            break
        img = Image.open(row["image_path"]).convert("RGB")
        axes[idx].imshow(img)
        axes[idx].set_title(f"score={row['score']:.2f}\nlabel={int(row['label'])}")
        axes[idx].axis("off")
    for idx in range(len(pires), len(axes)):
        axes[idx].axis("off")
    plt.suptitle("Exemples les plus mal classés")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig6_analyse_erreurs.png")


results_df = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
fig1_barplot_auroc_regime(results_df)
fig2_barplot_aupro_regime(results_df)
fig3_heatmap_auroc(results_df)
for regime in REGIMES:
    fig4_barplot_auroc_par_graine(results_df, regime)
for categorie in CATEGORIES:
    fig5_grille_exemples(categorie, SYNTH_DIR, DATA_DIR)
pred_csv = os.path.join(RESULTS_DIR, f"predictions_R1_reel_seul_{CATEGORIES[0]}_{SEEDS[0]}.csv")
if os.path.exists(pred_csv):
    fig6_analyse_erreurs(pred_csv)

Figure sauvegardée : ./figures/fig1_auroc_par_regime.png
Figure sauvegardée : ./figures/fig2_aupro_par_regime.png
Figure sauvegardée : ./figures/fig3_heatmap_auroc.png
Figure sauvegardée : ./figures/fig4_auroc_par_graine_R1_reel_seul.png
Figure sauvegardée : ./figures/fig4_auroc_par_graine_R2_aug_classique.png
Figure sauvegardée : ./figures/fig4_auroc_par_graine_R3_synthetique.png
Figure sauvegardée : ./figures/fig5_exemples_bottle.png
Figure sauvegardée : ./figures/fig5_exemples_cable.png
Figure sauvegardée : ./figures/fig5_exemples_transistor.png
Figure sauvegardée : ./figures/fig6_analyse_erreurs.png


## Cellule 8 — Analyse de quasi-duplication

In [19]:
def charger_modele_features():
    """Charge ResNet50 ImageNet sans la dernière couche, gelé."""
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(DEVICE)


def extraire_features_images(liste_chemins, model, device, batch_size=32):
    """Extrait les features ResNet50 pour une liste d'images."""
    preprocess = torchvision.transforms.Compose([
        torchvision.transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    features = []
    for i in tqdm(range(0, len(liste_chemins), batch_size), desc="Extraction features"):
        batch_paths = liste_chemins[i:i + batch_size]
        tensors = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            tensors.append(preprocess(img))
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            out = model(batch).squeeze()
        features.append(out.cpu().numpy())
    return np.vstack(features)


def lister_defauts_reels_test(data_dir, categorie):
    """Retourne les chemins des vrais défauts du split test (galerie de référence)."""
    test_dir = Path(data_dir, categorie, "test")
    chemins = []
    for d in test_dir.iterdir():
        if d.is_dir() and d.name != "good":
            chemins.extend([p for p in d.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
    print(f"Défauts réels test {categorie} : {len(chemins)} images")
    return chemins


def lister_images_synthetiques(synth_dir, categorie):
    """Retourne les chemins des images synthétiques."""
    chemin = Path(synth_dir, categorie)
    if not chemin.exists():
        return []
    chemins = [p for p in chemin.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
    return chemins


def analyser_quasi_duplication(features_defauts, features_synth, seuils):
    """Calcule distances cosinus au plus proche vrai défaut et taux sous seuils."""
    nn = NearestNeighbors(n_neighbors=1, metric="cosine")
    nn.fit(features_defauts)
    distances, indices = nn.kneighbors(features_synth)
    distances = distances.flatten()
    indices = indices.flatten()
    taux = {s: float(np.mean(distances < s)) for s in seuils}
    return distances, indices, taux


def fig7_histogramme_distances(distances, categorie):
    """Histogramme des distances au plus proche vrai défaut."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(distances, bins=50, edgecolor="black")
    ax.set_xlabel("Distance cosinus au plus proche défaut réel")
    ax.set_ylabel("Nombre d'images synthétiques")
    ax.set_title(f"Quasi-duplication — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig7_histogramme_distances_{categorie}.png")


def fig8_taux_quasi_duplication(taux_dict, categorie):
    """Barplot des taux d'images synthétiques sous chaque seuil."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([str(s) for s in taux_dict.keys()], [v * 100 for v in taux_dict.values()])
    ax.set_xlabel("Seuil de distance cosinus")
    ax.set_ylabel("% d'images synthétiques")
    ax.set_title(f"Taux de quasi-duplication — {categorie}")
    ax.set_ylim([0, 100])
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig8_taux_quasi_duplication_{categorie}.png")


def fig9_exemples_voisins(chemins_synth, chemins_defauts, indices, distances, categorie, n=10):
    """Affiche côte à côte synthétiques et leur plus proche voisin réel."""
    n = min(n, len(chemins_synth))
    fig, axes = plt.subplots(n, 2, figsize=(4, n * 2))
    for i in range(n):
        img_synth = Image.open(chemins_synth[i]).convert("RGB")
        img_voisin = Image.open(chemins_defauts[indices[i]]).convert("RGB")
        axes[i, 0].imshow(img_synth)
        axes[i, 0].set_title("Synthétique")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(img_voisin)
        axes[i, 1].set_title(f"Vrai défaut (dist={distances[i]:.3f})")
        axes[i, 1].axis("off")
    plt.suptitle(f"Voisinage — {categorie}")
    plt.tight_layout()
    sauvegarder_figure(fig, f"fig9_exemples_voisins_{categorie}.png")


model_features = charger_modele_features()
seuils = [0.05, 0.10, 0.15]

for categorie in CATEGORIES:
    chemins_defauts = lister_defauts_reels_test(DATA_DIR, categorie)
    chemins_synth = lister_images_synthetiques(SYNTH_DIR, categorie)
    if not chemins_synth or not chemins_defauts:
        print(f"Skip {categorie} : pas assez d'images")
        continue

    feats_defauts = extraire_features_images(chemins_defauts, model_features, DEVICE)
    feats_synth = extraire_features_images(chemins_synth, model_features, DEVICE)
    distances, indices, taux = analyser_quasi_duplication(feats_defauts, feats_synth, seuils)

    distances_csv = os.path.join(RESULTS_DIR, f"distances_quasi_duplication_{categorie}.csv")
    pd.DataFrame({
        "synthetique": [str(p) for p in chemins_synth],
        "voisin_reel": [str(chemins_defauts[i]) for i in indices],
        "distance": distances
    }).to_csv(distances_csv, index=False)
    print(f"Distances sauvegardées : {distances_csv}")
    print(f"Taux sous seuils : {taux}")

    fig7_histogramme_distances(distances, categorie)
    fig8_taux_quasi_duplication(taux, categorie)
    fig9_exemples_voisins(chemins_synth, chemins_defauts, indices, distances, categorie)

Défauts réels test bottle : 63 images


Extraction features: 100%|██████████| 4/4 [00:00<00:00,  9.01it/s]


Distances sauvegardées : ./results/distances_quasi_duplication_bottle.csv
Taux sous seuils : {0.05: 0.975, 0.1: 1.0, 0.15: 1.0}
Figure sauvegardée : ./figures/fig7_histogramme_distances_bottle.png
Figure sauvegardée : ./figures/fig8_taux_quasi_duplication_bottle.png
Figure sauvegardée : ./figures/fig9_exemples_voisins_bottle.png
Défauts réels test cable : 92 images


Extraction features: 100%|██████████| 4/4 [00:04<00:00,  1.08s/it]


Distances sauvegardées : ./results/distances_quasi_duplication_cable.csv
Taux sous seuils : {0.05: 0.7333333333333333, 0.1: 0.9916666666666667, 0.15: 1.0}
Figure sauvegardée : ./figures/fig7_histogramme_distances_cable.png
Figure sauvegardée : ./figures/fig8_taux_quasi_duplication_cable.png
Figure sauvegardée : ./figures/fig9_exemples_voisins_cable.png
Défauts réels test transistor : 40 images


Extraction features: 100%|██████████| 4/4 [00:00<00:00,  8.62it/s]


Distances sauvegardées : ./results/distances_quasi_duplication_transistor.csv
Taux sous seuils : {0.05: 0.8833333333333333, 0.1: 1.0, 0.15: 1.0}
Figure sauvegardée : ./figures/fig7_histogramme_distances_transistor.png
Figure sauvegardée : ./figures/fig8_taux_quasi_duplication_transistor.png
Figure sauvegardée : ./figures/fig9_exemples_voisins_transistor.png


## Cellule 9 — Ablations

## Cellule 10 — Rapport automatique

In [20]:
def generer_rapport(results_csv, summary_csv, ablations_aug_csv, ablations_synth_csv, figures_dir, output_md):
    """Génère README_results.md avec tableaux et liste des figures."""
    results = pd.read_csv(results_csv)
    summary = pd.read_csv(summary_csv)
    abl_aug = pd.read_csv(ablations_aug_csv) if os.path.exists(ablations_aug_csv) else None
    abl_synth = pd.read_csv(ablations_synth_csv) if os.path.exists(ablations_synth_csv) else None
    figures = sorted(Path(figures_dir).glob("*.png"))

    lignes = ["# Rapport de résultats — IND-3\n"]
    lignes.append("## Résumé des métriques par régime\n")
    lignes.append(summary.to_markdown(index=False))
    lignes.append("\n\n")

    lignes.append("## Résultats bruts\n")
    lignes.append(results.to_markdown(index=False))
    lignes.append("\n\n")

    if abl_aug is not None:
        lignes.append("## Ablations data-aug (R2)\n")
        lignes.append(abl_aug.to_markdown(index=False))
        lignes.append("\n\n")
    if abl_synth is not None:
        lignes.append("## Ablations synthétiques (R3 filtré)\n")
        lignes.append(abl_synth.to_markdown(index=False))
        lignes.append("\n\n")

    lignes.append("## Note méthodologique\n")
    lignes.append("> Les images de défauts réels utilisées dans l'analyse de quasi-duplication proviennent **uniquement du split test** de MVTec AD. \n")
    lignes.append("> Elles servent de **galerie de référence** pour le calcul de distance (comme le FID) et ne sont **jamais utilisées** pour l'entraînement ni la sélection de modèle. \n")
    lignes.append("\n")

    lignes.append("## Figures générées\n")
    for fig in figures:
        lignes.append(f"- {fig.name}")

    with open(output_md, "w") as f:
        f.write("\n".join(lignes))
    print(f"Rapport généré : {output_md}")


generer_rapport(
    os.path.join(RESULTS_DIR, "results.csv"),
    os.path.join(RESULTS_DIR, "summary.csv"),
    os.path.join(RESULTS_DIR, "ablations_aug.csv"),
    os.path.join(RESULTS_DIR, "ablations_synth.csv"),
    FIGURES_DIR,
    os.path.join(RESULTS_DIR, "README_results.md")
)

Rapport généré : ./results/README_results.md


## Cellule 11 — Analyse complémentaire (protocole expérimental)

Gain relatif R3 vs R2, AUROC par type de défaut (classes rares), FID, couverture et taux de quasi-duplication. Régénère le rapport final.

In [21]:
import glob
import re

from sklearn.metrics import roc_auc_score
from torchvision import models as tv_models
from torchvision import transforms as tv_transforms
from scipy import linalg

# ===== Constantes analyse complémentaire =====
SEUIL_DUP = 0.10
SEUILS_COUVERTURE = [0.05, 0.10, 0.15]
CHEMIN_METRIQUES_GEN = os.path.join(RESULTS_DIR, "generative_metrics.csv")
CHEMIN_GAIN = os.path.join(RESULTS_DIR, "gain_relatif.csv")
CHEMIN_CLASSES = os.path.join(RESULTS_DIR, "auroc_par_defaut.csv")
print("Analyse complémentaire — seuil duplication =", SEUIL_DUP)


def calculer_gain_relatif(df):
    """Gain R3 vs R2 par catégorie (moyenne +/- std sur les graines)."""
    metriques = ["auroc_image", "auroc_pixel", "aupro"]
    piv = df.pivot_table(index=["categorie", "seed"], columns="regime", values=metriques)
    lignes = []
    for cat in CATEGORIES:
        if cat not in piv.index.get_level_values(0):
            continue
        sub = piv.xs(cat, level="categorie")
        for m in metriques:
            col_r2 = (m, "R2_aug_classique")
            col_r3 = (m, "R3_synthetique")
            if col_r2 not in sub.columns or col_r3 not in sub.columns:
                continue
            r2, r3 = sub[col_r2], sub[col_r3]
            valides = r2.notna() & r3.notna()
            r2v, r3v = r2[valides], r3[valides]
            if len(r2v) == 0:
                continue
            delta = r3v - r2v
            gain_pct = (100 * delta / r2v).replace([np.inf, -np.inf], np.nan)
            lignes.append({
                "categorie": cat, "metrique": m,
                "R2_mean": r2v.mean(), "R3_mean": r3v.mean(),
                "delta_mean": delta.mean(), "delta_std": delta.std(ddof=0),
                "gain_relatif_pct": gain_pct.mean(), "n_graines": len(delta),
            })
    return pd.DataFrame(lignes)


def fig_gain_relatif(gain):
    """Barplot du gain R3 - R2 sur l'AUROC image par catégorie."""
    sub = gain[gain["metrique"] == "auroc_image"].set_index("categorie")
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    couleurs = ["#2a9d8f" if v >= 0 else "#e76f51" for v in sub["delta_mean"]]
    ax.bar(sub.index, sub["delta_mean"], yerr=sub["delta_std"], color=couleurs, capsize=5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Delta AUROC image (R3 - R2)")
    ax.set_title("Gain relatif du synthetique vs augmentation classique")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig11_gain_relatif.png")


def auroc_par_type(predictions_csv):
    """Calcule l'AUROC image par type de defaut (vs images good)."""
    df = pd.read_csv(predictions_csv)
    df["type_defaut"] = df["image_path"].apply(
        lambda p: Path(p).parent.name if Path(p).parent.name != "good" else "good"
    )
    good = df[df["type_defaut"] == "good"]["score"].values
    lignes = []
    for t in sorted(df["type_defaut"].unique()):
        if t == "good":
            continue
        scores = df[df["type_defaut"] == t]["score"].values
        y = np.r_[np.ones(len(scores)), np.zeros(len(good))]
        s = np.r_[scores, good]
        try:
            auc = roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan
        except Exception:
            auc = np.nan
        lignes.append({"type_defaut": t, "n_defaut": len(scores), "auroc": auc})
    return pd.DataFrame(lignes)


def table_auroc_classes():
    """Agrege l'AUROC par type de defaut pour tous les regimes/graines."""
    motif = re.compile(r"predictions_(.+?)_(bottle|cable|transistor)_(\d+)\.csv$")
    lignes = []
    for chemin in sorted(glob.glob(os.path.join(RESULTS_DIR, "predictions_*.csv"))):
        m = motif.search(Path(chemin).name)
        if not m:
            continue
        regime, cat, seed = m.group(1), m.group(2), int(m.group(3))
        for _, row in auroc_par_type(chemin).iterrows():
            lignes.append({"categorie": cat, "regime": regime, "seed": seed,
                           "type_defaut": row["type_defaut"],
                           "n_defaut": int(row["n_defaut"]), "auroc": row["auroc"]})
    return pd.DataFrame(lignes)


def fig_auroc_classes_rares(df_classes):
    """AUROC par type de defaut (R3) - met en avant les classes rares."""
    if df_classes.empty:
        return
    sub = df_classes[df_classes["regime"] == "R3_synthetique"]
    if sub.empty:
        sub = df_classes[df_classes["regime"] == "R1_reel_seul"]
    if sub.empty:
        print("[!] Pas de predictions pour tracer les classes rares")
        return
    agg = sub.groupby(["categorie", "type_defaut"]).agg(
        auroc=("auroc", "mean"), n=("n_defaut", "max")
    ).reset_index()
    agg["label"] = agg["categorie"] + " / " + agg["type_defaut"]
    agg = agg.sort_values("n")
    fig, ax = plt.subplots(figsize=(8, max(4, 0.45 * len(agg))))
    ax.barh(agg["label"], agg["auroc"])
    ax.set_xlim([0, 1])
    ax.set_xlabel("AUROC image par type de defaut")
    ax.set_title("Classes rares - AUROC par type de defaut")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig12_auroc_classes_rares.png")


def charger_inception():
    """Charge InceptionV3 ImageNet sans la couche de classification."""
    model = tv_models.inception_v3(weights=tv_models.Inception_V3_Weights.DEFAULT)
    model.fc = torch.nn.Identity()
    model.eval()
    return model.to(DEVICE)


def extraire_features_inception(chemins, model, batch_size=32):
    """Extrait les features Inception (2048) pour une liste d'images."""
    preprocess = tv_transforms.Compose([
        tv_transforms.Resize((299, 299)),
        tv_transforms.ToTensor(),
        tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    sorties = []
    for i in range(0, len(chemins), batch_size):
        batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in chemins[i:i + batch_size]])
        with torch.no_grad():
            sorties.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.vstack(sorties)


def calculer_fid(feats_reel, feats_synth):
    """Distance de Frechet entre features reelles et synthetiques."""
    mu1, mu2 = feats_reel.mean(0), feats_synth.mean(0)
    c1 = np.cov(feats_reel, rowvar=False)
    c2 = np.cov(feats_synth, rowvar=False)
    d = feats_reel.shape[1]
    c1 = c1 + np.eye(d) * 1e-6
    c2 = c2 + np.eye(d) * 1e-6
    res = linalg.sqrtm(c1 @ c2, disp=False)
    covmean = res[0] if isinstance(res, tuple) else res
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float((mu1 - mu2) @ (mu1 - mu2) + np.trace(c1 + c2 - 2 * covmean))


def calculer_couverture(chemins_reels, chemins_synth, seuils):
    """Fraction de vrais defauts couverts par au moins un synthetique proche."""
    f_reel = extraire_features_images(chemins_reels, model_features, DEVICE)
    f_synth = extraire_features_images(chemins_synth, model_features, DEVICE)
    nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(f_synth)
    distances, _ = nn.kneighbors(f_reel)
    distances = distances.flatten()
    return {s: float(np.mean(distances < s)) for s in seuils}, float(np.median(distances))


def construire_metriques_generatives():
    """FID + couverture (reel couvert par synth) + taux de quasi-duplication."""
    print("\n=== Metriques generatives (FID / couverture / duplication) ===")
    model_incep = charger_inception()
    lignes = []
    for cat in CATEGORIES:
        reels = lister_defauts_reels_test(DATA_DIR, cat)
        synth = lister_images_synthetiques(SYNTH_DIR, cat)
        if not reels or not synth:
            print(f"[!] {cat} : donnees manquantes, metriques ignorees")
            continue
        fid = calculer_fid(
            extraire_features_inception(reels, model_incep),
            extraire_features_inception(synth, model_incep),
        )
        couverture, dist_med = calculer_couverture(reels, synth, SEUILS_COUVERTURE)
        chemin_dist = os.path.join(RESULTS_DIR, f"distances_quasi_duplication_{cat}.csv")
        taux_dup = np.nan
        if os.path.exists(chemin_dist):
            d = pd.read_csv(chemin_dist)["distance"].values
            taux_dup = float(np.mean(d < SEUIL_DUP))
        lignes.append({
            "categorie": cat, "n_reels": len(reels), "n_synth": len(synth),
            "fid": fid,
            "couverture_0.05": couverture[0.05],
            "couverture_0.10": couverture[0.10],
            "couverture_0.15": couverture[0.15],
            "distance_mediane_reel_vers_synth": dist_med,
            "taux_quasi_duplication": taux_dup,
        })
        print(f"  {cat}: FID={fid:.2f} | couverture@0.15={couverture[0.15]:.2%} | dup@{SEUIL_DUP}={taux_dup:.2%}")
    return pd.DataFrame(lignes)


def fig_metriques_generatives(df_met):
    """Deux panneaux : FID et couverture par categorie."""
    if df_met.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(df_met["categorie"], df_met["fid"], color="#264653")
    axes[0].set_ylabel("FID (bas = plus proche du reel)")
    axes[0].set_title("FID defauts reels vs synthetiques")
    axes[1].bar(df_met["categorie"], df_met["couverture_0.15"] * 100, color="#2a9d8f")
    axes[1].set_ylabel("% de vrais defauts couverts")
    axes[1].set_ylim([0, 100])
    axes[1].set_title("Couverture (seuil cosinus 0.15)")
    plt.tight_layout()
    sauvegarder_figure(fig, "fig13_fid_couverture.png")


results_df = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
results_df["seed"] = results_df["seed"].astype(int)

gain_df = calculer_gain_relatif(results_df)
gain_df.to_csv(CHEMIN_GAIN, index=False)
print("\n=== Gain relatif R3 vs R2 ===")
print(gain_df.to_string(index=False))
fig_gain_relatif(gain_df)

classes_df = table_auroc_classes()
if not classes_df.empty:
    classes_df.to_csv(CHEMIN_CLASSES, index=False)
    print("\n=== AUROC par type de defaut (apercu) ===")
    print(classes_df.groupby(["categorie", "type_defaut"])["auroc"].mean().to_string())
    fig_auroc_classes_rares(classes_df)

met_df = construire_metriques_generatives()
if not met_df.empty:
    met_df.to_csv(CHEMIN_METRIQUES_GEN, index=False)
    print("\n=== Metriques generatives ===")
    print(met_df.to_string(index=False))
    fig_metriques_generatives(met_df)

generer_rapport(
    os.path.join(RESULTS_DIR, "results.csv"),
    os.path.join(RESULTS_DIR, "summary.csv"),
    os.path.join(RESULTS_DIR, "ablations_aug.csv"),
    os.path.join(RESULTS_DIR, "ablations_synth.csv"),
    FIGURES_DIR,
    os.path.join(RESULTS_DIR, "README_results.md"),
)
print("\nAnalyse complementaire terminee.")


Analyse complémentaire — seuil duplication = 0.1

=== Gain relatif R3 vs R2 ===
 categorie    metrique  R2_mean  R3_mean  delta_mean  delta_std  gain_relatif_pct  n_graines
    bottle auroc_image 0.994180 1.000000    0.005820   0.001631          0.585685          3
    bottle auroc_pixel 0.976776 0.985604    0.008827   0.001526          0.903977          3
    bottle       aupro 0.907535 0.944890    0.037355   0.005536          4.119916          3
     cable auroc_image 0.975512 0.987319    0.011807   0.002668          1.210019          3
     cable auroc_pixel 0.972203 0.984854    0.012651   0.000662          1.301364          3
     cable       aupro 0.913901 0.930979    0.017078   0.000958          1.868904          3
transistor auroc_image 0.942778 0.994167    0.051389   0.004619          5.452629          3
transistor auroc_pixel 0.920809 0.973227    0.052419   0.003997          5.694421          3
transistor       aupro 0.896195 0.946906    0.050712   0.004219          5.660469  

Extraction features: 100%|██████████| 4/4 [00:00<00:00,  8.17it/s]


  bottle: FID=41.10 | couverture@0.15=100.00% | dup@0.1=100.00%
Défauts réels test cable : 92 images


Extraction features: 100%|██████████| 4/4 [00:00<00:00,  7.19it/s]


  cable: FID=64.30 | couverture@0.15=100.00% | dup@0.1=99.17%
Défauts réels test transistor : 40 images


Extraction features: 100%|██████████| 4/4 [00:00<00:00,  8.28it/s]


  transistor: FID=71.19 | couverture@0.15=100.00% | dup@0.1=100.00%

=== Metriques generatives ===
 categorie  n_reels  n_synth       fid  couverture_0.05  couverture_0.10  couverture_0.15  distance_mediane_reel_vers_synth  taux_quasi_duplication
    bottle       63      120 41.098699         0.984127          1.00000              1.0                          0.015825                1.000000
     cable       92      120 64.304099         0.739130          0.98913              1.0                          0.042238                0.991667
transistor       40      120 71.194904         1.000000          1.00000              1.0                          0.030030                1.000000
Figure sauvegardée : ./figures/fig13_fid_couverture.png
Rapport généré : ./results/README_results.md

Analyse complementaire terminee.
